In [1]:
%%html
<style>
.cell-output-ipywidget-background {
    background-color: transparent !important;
}
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}  
</style>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tqdm.notebook import tqdm
from torch import optim
from typing import Tuple
from torch.distributions import Categorical

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from cube import Cube, moves, moves_idx
from cube_nn import cube_to_tensor_one_hot

# Constants of the environment
N_STATE = 54
N_STATE_TORCH = N_STATE*6  # one-hot encoding of the cube state
N_ACTION = 18


class MoveModel(nn.Module):
    def __init__(self):
        super().__init__()

        # Define actor's model
        self.actor = nn.Sequential(
            nn.Linear(N_STATE_TORCH, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Linear(64, N_ACTION)
        )

    def forward(self, obs):
        logit = self.actor(obs)
        return logit
    

class SuboptimalDataset(Dataset):
    def __init__(self, n_cubes, n_scrambling_moves=20):
        
        self.data = []
        for _ in tqdm(range(n_cubes), desc="Generating suboptimal dataset", leave=False):
            cube = Cube()
            for _ in range(n_scrambling_moves):
                move_idx = np.random.randint(len(moves))
                cube.move(moves[move_idx])
                state_tensor = cube_to_tensor_one_hot(cube).flatten().unsqueeze(0)
                action = torch.tensor(moves_idx[moves[move_idx].get_inverse()], dtype=torch.long)
                self.data.append((state_tensor, action))

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def train(model, dataloader, optimizer, n_epochs):
    model.train()
    criterion = nn.CrossEntropyLoss()
    losses = []
    pbar = tqdm(range(n_epochs), desc="Training epoch", leave=False)
    for epoch in pbar:
        total_loss = 0
        for states, actions in tqdm(dataloader, desc="Batches", leave=False):
            optimizer.zero_grad()
            logits = model(states)
            loss = criterion(logits.squeeze(1), actions)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(dataloader)
        losses.append(avg_loss)
        pbar.set_postfix({"Loss": f"{avg_loss:.4f}"})
    return losses
    


def try_to_solve(model, cube: Cube, max_moves: int = 40) -> Tuple[bool, int]:
    """
    Try to solve the cube with the current policy.
    Returns a tuple (solved: bool, n_moves: int)
    """
    def get_move(cube: Cube):
        cube_tensor = cube_to_tensor_one_hot(cube).flatten().unsqueeze(0)
        logit = model(cube_tensor)
        dist = Categorical(logits=logit)
        action = int(torch.argmax(dist.probs).item())
        move = moves[action]
        return move
    
    cube = cube.copy()
    iter_count = 0

    while not cube.is_solved() and iter_count < max_moves - 1:
        iter_count += 1

        move = get_move(cube)
        cube.move(move)

    return cube.is_solved(), iter_count

def evaluate(model, n_cubes: int, n_scrambling_moves: int, max_moves: int = 40) -> float:
    """
    Evaluate the agent on n_cubes random cubes.
    Returns the success rate.
    """
    successes = 0
    for _ in range(n_cubes):
        cube = Cube()
        cube.scramble(n_scrambling_moves)
        solved, n_moves = try_to_solve(model, cube, max_moves)
        if solved:
            successes += 1
    return successes / n_cubes

def full_eval(model):
    succ_rates = []
    pbar = tqdm(range(1, 20, 2), desc="Full evaluation", leave=False)
    for n_scrambling_moves in pbar:
        success_rate = evaluate(model, 50, n_scrambling_moves=n_scrambling_moves, max_moves=40)
        succ_rates.append(success_rate)
        pbar.set_postfix({"Success rate": f"{success_rate:.2f}"})

    return succ_rates

def full_train(model):
    succ_rates = [full_eval(model)]
    i = 0
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    while succ_rates[-1][-1] < 0.9:
        dataset = SuboptimalDataset(5000, n_scrambling_moves=20)
        dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
        losses = train(model, dataloader, optimizer, n_epochs=5)
        succ_rates.append(full_eval(model))
        fig, ax = plt.subplots(1, 2, figsize=(12, 5))
        ax[0].plot(losses)
        ax[0].set_xlabel("Epoch")
        ax[0].set_ylabel("Loss")
        # set fig title
        fig.suptitle(f"Training iteration {i}")
        ax[1].plot(range(1, 20, 2), succ_rates[-1])
        ax[1].set_xlabel("Number of scrambling moves")
        ax[1].set_ylabel("Success rate")
        ax[1].set_ylim(0, 1)
        # tight layout
        fig.tight_layout()
        # save figure
        fig.savefig(f"figs/training_iteration_{i}.png")
        i += 1


QSocketNotifier: Can only be used with threads started with QThread


In [3]:
move_model = MoveModel()

In [4]:
full_train(move_model)

Full evaluation:   0%|          | 0/10 [00:00<?, ?it/s]

Generating suboptimal dataset:   0%|          | 0/5000 [00:00<?, ?it/s]

Training epoch:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

...

Generating suboptimal dataset:   0%|          | 0/5000 [00:00<?, ?it/s]

Training epoch:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]